In [5]:
import torch
from spikingjelly.activation_based.encoding import PoissonEncoder
from torchvision.datasets import MNIST, CIFAR10
from torchvision.transforms import Compose, ToTensor, Normalize



In [8]:
# load CIFAR10 dataset
cifar10 = CIFAR10(root='../data', train=True, download=True, transform=ToTensor())

# load all images from CIFAR10 dataset
images = torch.stack([img[0] for img in cifar10])
# compute min and max values for normalization
min_val = images.min()
max_val = images.max()
print(f"Min value: {min_val}, Max value: {max_val}")

Files already downloaded and verified
Min value: 0.0, Max value: 1.0


In [11]:
# load MNIST dataset
mnist = MNIST(root='../data', train=True, download=True, transform=ToTensor())

# load all images from MNIST dataset
images = torch.stack([img[0] for img in mnist])
# compute min and max values for normalization
min_val = images.min()
max_val = images.max()
print(f"Min value: {min_val}, Max value: {max_val}")

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:03<00:00, 3.21MB/s]


Extracting ../data/MNIST/raw/train-images-idx3-ubyte.gz to ../data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 248kB/s]


Extracting ../data/MNIST/raw/train-labels-idx1-ubyte.gz to ../data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:01<00:00, 1.03MB/s]


Extracting ../data/MNIST/raw/t10k-images-idx3-ubyte.gz to ../data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 3.75MB/s]


Extracting ../data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ../data/MNIST/raw

Min value: 0.0, Max value: 1.0


In [9]:
cifar10[0][0]

tensor([[[0.2314, 0.1686, 0.1961,  ..., 0.6196, 0.5961, 0.5804],
         [0.0627, 0.0000, 0.0706,  ..., 0.4824, 0.4667, 0.4784],
         [0.0980, 0.0627, 0.1922,  ..., 0.4627, 0.4706, 0.4275],
         ...,
         [0.8157, 0.7882, 0.7765,  ..., 0.6275, 0.2196, 0.2078],
         [0.7059, 0.6784, 0.7294,  ..., 0.7216, 0.3804, 0.3255],
         [0.6941, 0.6588, 0.7020,  ..., 0.8471, 0.5922, 0.4824]],

        [[0.2431, 0.1804, 0.1882,  ..., 0.5176, 0.4902, 0.4863],
         [0.0784, 0.0000, 0.0314,  ..., 0.3451, 0.3255, 0.3412],
         [0.0941, 0.0275, 0.1059,  ..., 0.3294, 0.3294, 0.2863],
         ...,
         [0.6667, 0.6000, 0.6314,  ..., 0.5216, 0.1216, 0.1333],
         [0.5451, 0.4824, 0.5647,  ..., 0.5804, 0.2431, 0.2078],
         [0.5647, 0.5059, 0.5569,  ..., 0.7216, 0.4627, 0.3608]],

        [[0.2471, 0.1765, 0.1686,  ..., 0.4235, 0.4000, 0.4039],
         [0.0784, 0.0000, 0.0000,  ..., 0.2157, 0.1961, 0.2235],
         [0.0824, 0.0000, 0.0314,  ..., 0.1961, 0.1961, 0.

In [5]:
from torchvision.transforms import Compose, ToTensor, Normalize
from torchvision.datasets import MNIST, CIFAR10

SINGLE_CHANNEL_DATASETS = ["MNIST", "FashionMNIST", "KMNIST"]


class MNISTRepeated(MNIST):
    def __init__(
        self, *args, repeat: int = 1, normalize: bool = True, **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.repeat = repeat
        self.normalize = normalize
        self.transform_pipeline = (
            Compose(
                [
                    ToTensor(),
                    Normalize((0.1307,), (0.3081,)),
                ]
            )
            if normalize
            else ToTensor()
        )
        if not self.normalize:
            print(
                "Z-score standarization is disabled, will use min max scaling."
            )

    def __getitem__(self, index):
        img, target = super().__getitem__(index)

        img_tensor = self.transform_pipeline(img).unsqueeze(0)
        if not self.normalize:
            img_tensor = (img_tensor - img_tensor.min()) / (
                img_tensor.max() - img_tensor.min()
            )
        img_tensor = img_tensor.repeat(self.repeat, 1, 1, 1)

        return img_tensor, target


class CIFAR10Repeated(CIFAR10):

    def __init__(
        self, *args, repeat: int = 1, normalize: bool = True, **kwargs
    ):

        super().__init__(*args, transform=None, **kwargs)

        self.repeat = repeat
        self.normalize = normalize
        self.transform_pipeline = (
            Compose(
                [
                    ToTensor(),
                    Normalize(
                        (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
                    ),
                ]
            )
            if normalize
            else ToTensor()
        )

        if not self.normalize:
            print(
                "Z-score standarization is disabled, will use min max scaling."
            )

    def __getitem__(self, index):
        img, target = super().__getitem__(index)

        img_tensor = self.transform_pipeline(img).unsqueeze(0)
        if not self.normalize:
            img_tensor = (img_tensor - img_tensor.min()) / (
                img_tensor.max() - img_tensor.min()
            )

        repeated_img_tensor = img_tensor.repeat(self.repeat, 1, 1, 1)

        return repeated_img_tensor, target


class DatasetFactory:
    @staticmethod
    def create_dataset(name, *args, **kwargs):
        if name == "MNIST":
            return MNISTRepeated(*args, **kwargs)
        elif name == "CIFAR10":
            return CIFAR10Repeated(*args, **kwargs)
        else:
            raise ValueError(f"Dataset {name} not recognized.")


In [ ]:
import torch
from torch import nn

from spikingjelly.activation_based import surrogate, neuron, layer
from spikingjelly.activation_based.model.sew_resnet import sew_resnet18
from spikingjelly.activation_based.model.spiking_vgg import spiking_vgg11_bn
from typing import Dict, Callable, Any


def SewResnet18(
    n_channels: int = 1,
    output_size: int = 10,
    neuron_model: neuron.BaseNode = neuron.LIFNode,
    surrogate_function: surrogate.SurrogateFunctionBase = surrogate.Sigmoid,
) -> nn.Module:
    net = sew_resnet18(
        pretrained=False,
        spiking_neuron=neuron_model,
        cnf="IAND",
        surrogate_function=surrogate_function(),
    )
    net.conv1 = layer.Conv2d(
        n_channels,
        64,
        kernel_size=(7, 7),
        stride=(2, 2),
        padding=(3, 3),
        bias=False,
    )
    net.fc = layer.Linear(512, output_size)
    return net


def SpikingVGG11BN(
    n_channels: int = 1,
    output_size: int = 10,
    neuron_model: neuron.BaseNode = neuron.LIFNode,
    surrogate_function: surrogate.SurrogateFunctionBase = surrogate.Sigmoid,
    remove_last_pool: int = 2 
) -> nn.Module:
    net = spiking_vgg11_bn(
        pretrained=False,
        spiking_neuron=neuron_model,
        surrogate_function=surrogate_function(),
    )

    net.features[0] = layer.Conv2d(
        n_channels,
        64,
        kernel_size=(3, 3),
        stride=(1, 1),
        padding=(1, 1),
        bias=False,
    )
    net.classifier[6] = layer.Linear(4096, output_size)

    pool_indices: list[int] = [i for i, module in enumerate(net.features) if isinstance(module, nn.MaxPool2d)]

    if remove_last_pool > 0 and len(pool_indices) >= remove_last_pool:
        modules_to_keep = []
        last_pool_index_to_keep = pool_indices[len(pool_indices) - remove_last_pool] if remove_last_pool < len(pool_indices) else -1

        for i, module in enumerate(net.features):
            if i <= last_pool_index_to_keep or not isinstance(module, nn.MaxPool2d):
                modules_to_keep.append(module)


        net.features = nn.Sequential(*modules_to_keep)
        
    return net


MODEL_MAP: Dict[str, Callable[[Any], nn.Module]] = {
    "sew_resnet": SewResnet18,
    "spiking_vgg": SpikingVGG11BN,
}


In [55]:
from spikingjelly.activation_based import functional


vgg = MODEL_MAP["spiking_vgg"](
    n_channels=3,
    output_size=10,
    neuron_model=neuron.LIFNode,
    surrogate_function=surrogate.Sigmoid,
    remove_last_pool=2  
)
functional.set_step_mode(vgg, "m")

In [56]:

mnist_dataset = DatasetFactory.create_dataset(
    "CIFAR10",
    root="../data",
    repeat=10
)

In [61]:
first_img = mnist_dataset[0][0]
vgg(first_img.unsqueeze(1)).mean(0)

tensor([[ 0.0034, -0.0103,  0.0071, -0.0008, -0.0121,  0.0052,  0.0112,  0.0087,
         -0.0085,  0.0014]], grad_fn=<MeanBackward1>)